In [0]:
import requests
import json
import os

In [0]:
# 1. Petición a la API general de FPL
url = "https://fantasy.premierleague.com/api/bootstrap-static/"
headers = {"User-Agent": "Mozilla/5.0"} 
response = requests.get(url, headers=headers)

if response.status_code != 200:
    raise Exception(f"Error en la llamada a la API. Código HTTP: {response.status_code}")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.fpl_landing;

In [0]:
landing_path = "/Volumes/workspace/default/fpl_landing/bootstrap_static.json"

with open(landing_path, 'w', encoding='utf-8') as f:
    json.dump(response.json(), f)

print(f"JSON guardado en: {landing_path}")

In [0]:
# 1. Leer el JSON desde volume usando PySpark
df_raw = spark.read.option("multiline", "true").json("/Volumes/workspace/default/fpl_landing/bootstrap_static.json")

In [0]:
# 3. Escribir en formato Delta
(df_raw.write
    .format("delta")
    .mode("overwrite") 
    .option("mergeSchema", "true") # Permite que la tabla acepte nuevos campos
    .saveAsTable("workspace.default.bronze_fpl_data")
)

In [0]:
df_raw.printSchema()